# Data Preprocessing

In [1]:
# Load data from Excel file in data folder
import pandas as pd

cpds_raw_data = pd.read_excel('../data/pd_export_02_2025_targets_standardized.xlsx', sheet_name='COMPOUNDS')
targets_raw_data = pd.read_excel('../data/pd_export_02_2025_targets_standardized.xlsx', sheet_name='TARGETS')
external_ids_raw_data = pd.read_excel('../data/pd_export_02_2025_targets_standardized.xlsx', sheet_name='EXTERNAL IDS')


In [2]:
# Merge targets and compounds on pdid and keep source-specific probe columns.
raw_data = pd.merge(
    targets_raw_data,
    cpds_raw_data,
    on="pdid",
    how="left",
    suffixes=("_target", "_compound"),
)

In [3]:
print("Cpds data shape:", cpds_raw_data.shape)
print("Targets data shape:", targets_raw_data.shape)
print("Raw data shape:", raw_data.shape)

Cpds data shape: (138689, 38)
Targets data shape: (270054, 35)
Raw data shape: (270054, 72)


### Bioactivity table

In [4]:
import re

# Lookup table that flags where each annotation came from.
assay_type = pd.DataFrame(
    [
        {
            "assay_type": "biochemical",
            "source_column": "activity_biochemical",
            "description": "Annotation extracted from activity_biochemical"
        },
        {
            "assay_type": "cell",
            "source_column": "activity_cell",
            "description": "Annotation extracted from activity_cell"
        },
    ]
)

PLACEHOLDERS = {"-", "na", "n/a", "none", "nan", "not available", "not determined"}

def split_activity_annotations(value):
    """Split one activity field into separate annotations/rows."""
    if pd.isna(value):
        return []

    text = str(value).strip()
    if not text:
        return []

    parts = re.split(r"\s*(?:;|\||\n)+\s*", text)
    clean_parts = []
    for part in parts:
        p = part.strip()
        if p and p.lower() not in PLACEHOLDERS:
            clean_parts.append(p)

    return clean_parts

def parse_value(annotation):
    """Extract numeric value when present; unavailable fields remain empty."""
    val_match = re.search(r"(?:<=|>=|=|<|>)?\s*(-?\d+(?:\.\d+)?)", annotation)
    if val_match:
        return pd.to_numeric(val_match.group(1), errors="coerce")
    return None

def as_probe_flag(value):
    """Normalize heterogeneous probe encodings to 1/0/None."""
    if pd.isna(value):
        return None

    text = str(value).strip().lower()
    if text in {"1", "1.0", "true", "t", "yes", "y"}:
        return 1
    if text in {"0", "0.0", "false", "f", "no", "n", "-"}:
        return 0
    return None

def normalize_moa(value):
    """Convert placeholder MOA labels to missing values."""
    if pd.isna(value):
        return None

    text = str(value).strip()
    if text == "" or text.lower() in PLACEHOLDERS:
        return None

    return text

records = []

for _, row in raw_data.iterrows():
    # Keep complexes unsplit in bioactivity_table (e.g., "EHMT1,EHMT2").
    target_key = row.get("gene_name")

    if pd.isna(target_key) or str(target_key).strip() == "" or str(target_key).strip().lower() in PLACEHOLDERS:
        continue

    # Keep only probe-related target annotations; this removes off-target rows labeled as 0.
    probe_target_flag = as_probe_flag(row.get("probe_target"))
    if probe_target_flag != 1:
        continue

    moa_value = normalize_moa(row.get("moa"))

    for assay_col, assay_label in [("activity_biochemical", "biochemical"), ("activity_cell", "cell")]:
        annotations = split_activity_annotations(row.get(assay_col))

        for annotation in annotations:
            records.append(
                {
                    "inchikey": row.get("inchikey"),
                    "target_key": str(target_key).strip(),
                    "moa": moa_value,
                    "bioactivity_type": None,
                    "relation": None,
                    "value": parse_value(annotation),
                    "unit": None,
                    "assay_type": assay_label,
                    "assay_description": None,
                    "cell_line": None,
                    "concentration": None,
                    "concentration_unit": None,
                    "source_db": "Probes & Drugs",
                    "source": None,
                    "source_xref": None,
                    "xref_id": None,
                }
            )

bioactivity_table = pd.DataFrame.from_records(
    records,
    columns=[
        "inchikey",
        "target_key",
        "moa",
        "bioactivity_type",
        "relation",
        "value",
        "unit",
        "assay_type",
        "assay_description",
        "cell_line",
        "concentration",
        "concentration_unit",
        "source_db",
        "source",
        "source_xref",
        "xref_id",
    ],
)

print("assay_type shape:", assay_type.shape)
display(assay_type)

print("bioactivity_table shape:", bioactivity_table.shape)
display(bioactivity_table.head(20))

assay_type shape: (2, 3)


,assay_type,source_column,description
0,biochemical,activity_biochemical,Annotation extracted from activity_biochemical
1,cell,activity_cell,Annotation extracted from activity_cell


bioactivity_table shape: (6392, 16)


,inchikey,target_key,moa,bioactivity_type,relation,value,unit,assay_type,assay_description,cell_line,concentration,concentration_unit,source_db,source,source_xref,xref_id
0,RRZVGDGTWNQAPW-UHFFFAOYSA-N,BAZ2B,inhibitor;antagonist,None,None,6.77,None,biochemical,None,None,None,None,Probes & Drugs,None,None,None
1,RRZVGDGTWNQAPW-UHFFFAOYSA-N,BAZ2A,inhibitor;antagonist,None,None,6.96,None,biochemical,None,None,None,None,Probes & Drugs,None,None,None
2,WRUWGLUCNBMGPS-UHFFFAOYSA-N,BRD9,antagonist;binding agent,None,None,7.10,None,biochemical,None,None,None,None,Probes & Drugs,None,None,None
3,WRUWGLUCNBMGPS-UHFFFAOYSA-N,BRD9,antagonist;binding agent,None,None,6.95,None,cell,None,None,None,None,Probes & Drugs,None,None,None
4,QOECJCJVIMVJGX-UHFFFAOYSA-N,EHMT2,inhibitor,None,None,8.17,None,biochemical,None,None,None,None,Probes & Drugs,None,None,None
5,QOECJCJVIMVJGX-UHFFFAOYSA-N,EHMT2,inhibitor,None,None,7.16,None,cell,None,None,None,None,Probes & Drugs,None,None,None
6,QOECJCJVIMVJGX-UHFFFAOYSA-N,EHMT1,inhibitor,None,None,7.72,None,biochemical,None,None,None,None,Probes & Drugs,None,None,None
7,QOECJCJVIMVJGX-UHFFFAOYSA-N,EHMT1,inhibitor,None,None,7.89,None,cell,None,None,None,None,Probes & Drugs,None,None,None
8,QOECJCJVIMVJGX-UHFFFAOYSA-N,"EHMT1,EHMT2",None,None,None,7.09,None,cell,None,None,None,None,Probes & Drugs,None,None,None
9,LVDRREOUMKACNJ-BKMJKUGQSA-N,BRD9,inhibitor;antagonist,None,None,7.00,None,biochemical,None,None,None,None,Probes & Drugs,None,None,None


### Compound table

In [5]:
import re

# Build compound table from compounds raw data and enrich with ChEMBL from external IDs.
def normalize_chembl_values(values):
    cleaned = []
    for value in values:
        if pd.isna(value):
            continue
        parts = re.split(r"\s*(?:;|\||,)+\s*", str(value).strip())
        for part in parts:
            p = part.strip()
            if p and p != "-":
                cleaned.append(p)
    return ";".join(dict.fromkeys(cleaned)) if cleaned else None

def first_non_null(series):
    for value in series:
        if pd.notna(value) and str(value).strip() != "":
            return value
    return None

def as_probe_flag(value):
    if pd.isna(value):
        return None
    text = str(value).strip().lower()
    if text in {"1", "1.0", "true", "t", "yes", "y"}:
        return 1
    if text in {"0", "0.0", "false", "f", "no", "n", "-"}:
        return 0
    return None

cpd_with_ids = cpds_raw_data.copy()
cpd_with_ids["chembl_id"] = None

# Keep only compounds annotated as probes in the compounds sheet.
cpd_with_ids["probe_flag"] = cpd_with_ids.get("probe").map(as_probe_flag)
cpd_with_ids = cpd_with_ids[cpd_with_ids["probe_flag"] == 1].copy()

if "ChEMBL" in external_ids_raw_data.columns:
    merge_keys = [col for col in ["pdid", "name", "probe"] if col in cpd_with_ids.columns and col in external_ids_raw_data.columns]

    if merge_keys:
        ext_chembl = external_ids_raw_data[merge_keys + ["ChEMBL"]].copy()
        ext_chembl = ext_chembl.dropna(subset=["ChEMBL"])

        ext_chembl = (
            ext_chembl
            .groupby(merge_keys, dropna=False)["ChEMBL"]
            .apply(lambda s: normalize_chembl_values(s.tolist()))
            .reset_index(name="chembl_id")
        )

        cpd_with_ids = cpd_with_ids.merge(ext_chembl, on=merge_keys, how="left", suffixes=("", "_from_ext"))
        if "chembl_id_from_ext" in cpd_with_ids.columns:
            cpd_with_ids["chembl_id"] = cpd_with_ids["chembl_id_from_ext"]
            cpd_with_ids = cpd_with_ids.drop(columns=["chembl_id_from_ext"])

compound_table = (
    cpd_with_ids.groupby("inchikey", dropna=True, as_index=False)
    .agg(
        smiles=("smiles", first_non_null),
        chembl_id=("chembl_id", lambda s: normalize_chembl_values(s.tolist())),
        name=("name", first_non_null),
    )
)

compound_table = compound_table[["inchikey", "smiles", "chembl_id", "name"]]

print("compound_table shape:", compound_table.shape)
display(compound_table.head(20))

compound_table shape: (4840, 4)


,inchikey,smiles,chembl_id,name
0,AACFPJSJOWQNBN-UHFFFAOYSA-N,O=C1NCCCc2c1oc1ccc(O)cc21,CHEMBL1450770,CID755673
1,AAHNBILIYONQLX-QFIPXVFZSA-N,COc1cc(-c2cn([C@H]3CCc4c(F)cccc4N(CC(F)(F)F)C3...,CHEMBL3609749,GSM1
2,AAHNBILIYONQLX-UHFFFAOYSA-N,COc1cc(-c2cn(C3CCc4c(F)cccc4N(CC(F)(F)F)C3=O)n...,CHEMBL3609637,GSM1
3,AAISIAHCALLSAM-XBMOYTETSA-N,COC[C@H]1OC(=O)/C(=C/N2CCN(C3CCN(C)CC3)CC2)C2=...,CHEMBL271430,PD081027
4,AAIYXSVMDBVZTF-OQLLNIDSSA-N,N/N=C(\Cc1ccc(O)c(Br)c1)C(=O)NCCS,CHEMBL2047680,PD084089
5,AALSIMHTWOFTTB-UHFFFAOYSA-N,C#Cc1cccc(Nc2ncnc3cc(OCCOC(=O)C(C)c4ccc(CC(C)C...,CHEMBL3741026,PD080903
6,AARBAVYWWQCGFS-BXQOSRMJSA-N,COC[C@H]1OC(=O)/C(=C/N(C)CCCN2CCN(C)CC2)C2=C(O...,CHEMBL407214,PD083133
7,AAYTXOPXLXAKHP-UHFFFAOYSA-N,O=C(c1ccc(Cl)cc1)N1CCN(c2ccnc3cc(Cl)ccc23)CC1,CHEMBL1360505,ML189
8,AAZUPSFRSHFTGV-INIZCTEOSA-N,C[C@H](NC(=O)c1ccc2c(-c3ccc(C(F)(F)F)cc3)cccc2...,CHEMBL5198592,VT104
9,ABEGFBRZNZWQBS-UHFFFAOYSA-N,COc1cccc(Nc2[nH]nc3ncnc(Nc4cccc(Cl)c4)c23)c1,CHEMBL120979,PD082610


### Target table

In [6]:
# Build target table from targets raw data with unsplit target_key values.
type_candidates = ["type", "target_type", "target class", "target_class"]
type_col = next((col for col in type_candidates if col in targets_raw_data.columns), None)

if type_col is None:
    type_series = pd.Series([None] * len(targets_raw_data), index=targets_raw_data.index)
else:
    type_series = targets_raw_data[type_col]

def normalize_target_type(value):
    if pd.isna(value):
        return None

    text = str(value).strip()
    key = text.lower()

    type_map = {
        "single protein": "protein",
        "chimeric protein": "protein",
        "protein family": "family",
        "selectivity group": "family",
        "protein-protein interaction": "ppi",
        "protein complex": "complex",
        "protein complex group": "complex",
    }

    # Keep labels not explicitly mapped (e.g., selectivity group, chimeric protein).
    return type_map.get(key, text)

target_table = pd.DataFrame(
    {
        "target_key": targets_raw_data.get("gene_name"),
        "type": type_series.map(normalize_target_type),
        "name": targets_raw_data.get("target_name"),
    }
)

target_table["target_key"] = target_table["target_key"].astype(str).str.strip()

target_table = (
    target_table
    .replace({"target_key": {"": None, "-": None, "nan": None}})
    .dropna(subset=["target_key"])
    .reset_index(drop=True)
)

# Keep only targets that are present in bioactivity_table (target_key is the primary key).
if "bioactivity_table" not in globals():
    raise RuntimeError("bioactivity_table is not defined. Run the Bioactivity cell first.")

valid_target_keys = set(
    bioactivity_table["target_key"]
    .dropna()
    .astype(str)
    .str.strip()
    .tolist()
)

target_table = target_table[target_table["target_key"].isin(valid_target_keys)].reset_index(drop=True)

# Enforce a single row per target_key by keeping the first observed entry.
target_table = target_table.drop_duplicates(subset=["target_key"], keep="first").reset_index(drop=True)

print("target_table shape:", target_table.shape)
print("Unique target_key count:", target_table["target_key"].nunique())
print("Unique type values:", sorted(target_table["type"].dropna().unique().tolist()))
display(target_table.head(20))

target_table shape: (898, 3)
Unique target_key count: 898
Unique type values: ['complex', 'family', 'ppi', 'protein']


,target_key,type,name
0,CECR2,protein,Chromatin remodeling regulator CECR2
1,BAZ2B,protein,Bromodomain adjacent to zinc finger domain pro...
2,BAZ2A,protein,Bromodomain adjacent to zinc finger domain pro...
3,BRD7,protein,Bromodomain-containing protein 7
4,BRD9,protein,Bromodomain-containing protein 9
5,EP300,protein,Histone acetyltransferase p300
6,CREBBP,protein,CREB-binding protein
7,BRD4,protein,Bromodomain-containing protein 4
8,BRPF1,protein,Peregrin
9,KDM1A,protein,Lysine-specific histone demethylase 1A


### Uniprot table

In [7]:
# Build Uniprot table from targets raw data with one row per gene/uniprot pair.
def split_components(value):
    if pd.isna(value):
        return []

    text = str(value).strip()
    if not text:
        return []

    parts = [p.strip() for p in text.split(",") if p.strip()]
    return parts

uniprot_records = []

for _, row in targets_raw_data.iterrows():
    genes = split_components(row.get("gene_name"))
    uniprots = split_components(row.get("human_uniprot_id"))

    # Align gene and uniprot components by position for protein complexes.
    max_len = max(len(genes), len(uniprots))

    if max_len == 0:
        continue

    if len(genes) < max_len:
        genes = genes + [None] * (max_len - len(genes))
    if len(uniprots) < max_len:
        uniprots = uniprots + [None] * (max_len - len(uniprots))

    for i in range(max_len):
        target_key = genes[i]
        uniprot_id = uniprots[i]
        hgnc = genes[i]

        if target_key is None:
            continue

        uniprot_records.append(
            {
                "uniprot_id": uniprot_id,
                "target_key": target_key,
                "hgnc": hgnc,
                "species": "Homo sapiens",
            }
        )

uniprot_table = pd.DataFrame(
    uniprot_records,
    columns=["uniprot_id", "target_key", "hgnc", "species"],
)

uniprot_table = (
    uniprot_table
    .dropna(subset=["uniprot_id", "target_key"])
    .drop_duplicates(subset=["uniprot_id", "target_key", "hgnc"])
    .reset_index(drop=True)
)

# Keep only targets that are present in bioactivity_table (target_key is the primary key).
if "bioactivity_table" not in globals():
    raise RuntimeError("bioactivity_table is not defined. Run the Bioactivity cell first.")

valid_target_keys = set(
    bioactivity_table["target_key"]
    .dropna()
    .astype(str)
    .str.strip()
    .tolist()
)

uniprot_table = uniprot_table[uniprot_table["target_key"].isin(valid_target_keys)].reset_index(drop=True)

print("uniprot_table shape:", uniprot_table.shape)
display(uniprot_table.head(20))

uniprot_table shape: (1413, 4)


,uniprot_id,target_key,hgnc,species
0,Q9BXF3,CECR2,CECR2,Homo sapiens
1,Q9UIF8,BAZ2B,BAZ2B,Homo sapiens
2,Q9UIF9,BAZ2A,BAZ2A,Homo sapiens
3,Q9NPI1,BRD7,BRD7,Homo sapiens
4,Q9H8M2,BRD9,BRD9,Homo sapiens
5,Q09472,EP300,EP300,Homo sapiens
6,Q92793,CREBBP,CREBBP,Homo sapiens
7,O60885,BRD4,BRD4,Homo sapiens
8,P55201,BRPF1,BRPF1,Homo sapiens
9,O60341,KDM1A,KDM1A,Homo sapiens


In [8]:
from pathlib import Path

# Export all tables to TSV files.
output_dir = Path("../staging/probes_and_drugs")
output_dir.mkdir(parents=True, exist_ok=True)

bioactivity_path = output_dir / "bioactivity.tsv"
compound_path = output_dir / "compound.tsv"
target_path = output_dir / "target.tsv"
uniprot_path = output_dir / "uniprot.tsv"

bioactivity_table.to_csv(bioactivity_path, sep="\t", index=False)
compound_table.to_csv(compound_path, sep="\t", index=False)
target_table.to_csv(target_path, sep="\t", index=False)
uniprot_table.to_csv(uniprot_path, sep="\t", index=False)

print("Exported files:")
print(bioactivity_path)
print(compound_path)
print(target_path)
print(uniprot_path)

Exported files:
../staging/probes_and_drugs/bioactivity.tsv
../staging/probes_and_drugs/compound.tsv
../staging/probes_and_drugs/target.tsv
../staging/probes_and_drugs/uniprot.tsv
